# NovelForge - Note 3 : Deep Learning Avance

Objectif : tester une approche de Transfer Learning avec un Transformer leger HuggingFace pour la classification multilabel des genres.

Le notebook est volontairement configure sur un petit sous-echantillon afin de rester executable localement, y compris sur CPU.

## 1. Imports et configuration

Le code complexe est encapsule dans `src/transformer_model.py`. Le notebook charge les donnees, prepare les labels et lance un fine-tuning minimal de DistilBERT.

In [1]:
from pathlib import Path
import os
import sys
import time

import joblib
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from preprocessing import (
    TextPreprocessor,
    add_filtered_label_column,
    clean_dataframe,
    drop_columns_if_present,
    infer_column,
    parse_multilabel_cell,
    remove_synopsis_anomalies,
)
from project_config import GENRE_VOCABULARY
from transformer_model import NovelForgeTransformer, TransformerConfig

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

PROJECT_DIR

WindowsPath('C:/Users/ClémentPERRET/OneDrive - EQUATERRE-VDS/Bureau/Cours/DeepLearning')

## 2. Chargement et preprocessing

On reutilise la meme preparation que les jalons precedents : `description` comme texte, `tags` filtres en genres, suppression de `cover`, nettoyage robuste et suppression des descriptions trop courtes.

In [2]:
def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported dataset format: {suffix}")

raw_candidates = [
    Path(os.getenv("NOVELFORGE_DATASET", "")) if os.getenv("NOVELFORGE_DATASET") else None,
    PROJECT_DIR / "data" / "data.csv",
    PROJECT_DIR / "data" / "dataset.csv",
]
raw_path = next((path for path in raw_candidates if path is not None and path.exists()), None)
if raw_path is None:
    raise FileNotFoundError("No dataset found in data/.")

df_raw = load_dataset(raw_path)
df_raw = drop_columns_if_present(df_raw, ["cover"])

text_column = infer_column(df_raw.columns, ["synopsis", "summary", "description", "overview", "plot", "resume"])
label_column = infer_column(df_raw.columns, ["genres", "genre", "tags", "categories", "labels", "target"])

df_raw = add_filtered_label_column(df_raw, label_column, "genre_labels", GENRE_VOCABULARY)
preprocessor = TextPreprocessor(lowercase=True, remove_urls=True, lemmatize=True)
df = clean_dataframe(df_raw, text_column=text_column, clean_column="synopsis_clean", preprocessor=preprocessor)
df = remove_synopsis_anomalies(df, clean_column="synopsis_clean", min_words=5)
df = df[df["genre_labels"].str.len().gt(0)].copy()

print(f"Dataset: {raw_path}")
print(f"Rows after cleaning: {len(df):,}")
display(df[["title", "synopsis_clean", "genre_labels"]].head())

Dataset: C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\data\data.csv
Rows after cleaning: 69,511


,title,synopsis_clean,genre_labels
0,Salad Days (Tang LiuZang) - Part 2,the second season of salad day tang liuzang,"[BL, Romance, Shounen-ai, Sports]"
1,The Master of Diabolism,as the grandmaster who found the demonic sect wei wuxian roam the world in his wanton way hat by million for the chao he creat in the en...,"[Action, Adventure, BL, Comedy, Mystery, Romance, Shounen-ai, Martial Arts, Supernatural, Xianxia]"
2,JoJo's Bizarre Adventure Part 7: Steel Ball Run,set in steel ball run spotlight gyro zepelli and johnny joestar as they pit their spirit on a fifty million dollar race across the heart...,"[Action, Adventure, Horror, Mystery, Seinen, Historical]"
3,A Sign of Affection,yuki is a typical college student whose world revolve around her friend social media and the latest sale but when a chance encounter on ...,"[Romance, Shoujo, Slice of Life]"
4,Moriarty the Patriot,before he was sherlock s rival moriarty fought against the unfair class caste system in london by mak sure corrupt nobility got their co...,"[Mystery, Shounen, Historical]"


## Jalon 8 - Pourquoi un Transformer / Transfer Learning ?

La baseline TF-IDF est forte car beaucoup de genres sont correles a des mots-cles. Mais elle ne comprend pas le contexte : les n-grammes restent locaux et l'ordre global est peu modelise. Le LSTM ajoute la sequentialite, mais notre experience montre qu'un RNN fondamental est couteux, sensible au desequilibre multilabel et difficile a optimiser sur CPU.

Un Transformer repose sur le mecanisme d'attention : chaque token peut ponderer directement les autres tokens de la sequence. En pratique, l'attention apprend des dependances longues sans parcourir le texte uniquement de gauche a droite comme un RNN. Avec le Transfer Learning, DistilBERT part de representations linguistiques deja apprises sur de tres grands corpus. Le fine-tuning adapte seulement ces representations a notre taxonomie de genres, ce qui est plus efficace que d'apprendre des embeddings et des dependances depuis zero.

un Transformer compact de type BERT/DistilBERT est choisi comme compromis ingenierie : il est plus leger que BERT, disponible dans HuggingFace, compatible CPU pour une demonstration locale, et suffisant pour prouver la technologie de pointe attendue au Jalon 8.

## 3. Sous-echantillon et encodage multilabel

Le fine-tuning Transformer complet serait long sur CPU. On prend donc un petit sous-echantillon reproductible pour demontrer le pipeline, puis on evalue avec les memes metriques F1 micro/macro.

In [3]:
TEXT_COLUMN = "synopsis_clean"
LABEL_COLUMN = "genre_labels"

model_df = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
model_df[LABEL_COLUMN] = model_df[LABEL_COLUMN].apply(parse_multilabel_cell)
model_df = model_df[model_df[LABEL_COLUMN].str.len().gt(0)].copy()

genre_counts = model_df[LABEL_COLUMN].explode().value_counts()
MIN_GENRE_FREQ = 100
kept_genres = set(genre_counts[genre_counts >= MIN_GENRE_FREQ].index)
model_df[LABEL_COLUMN] = model_df[LABEL_COLUMN].apply(lambda labels: [label for label in labels if label in kept_genres])
model_df = model_df[model_df[LABEL_COLUMN].str.len().gt(0)].copy()

MAX_ROWS = 240
if len(model_df) > MAX_ROWS:
    model_df = model_df.sample(n=MAX_ROWS, random_state=42).copy()

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(model_df[LABEL_COLUMN]).astype("float32")
X = model_df[TEXT_COLUMN].fillna("").astype(str).tolist()

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

id2label = {index: label for index, label in enumerate(mlb.classes_)}
label2id = {label: index for index, label in id2label.items()}

print(f"Rows used: {len(model_df):,}")
print(f"Train: {len(X_train):,} | Valid: {len(X_valid):,} | Test: {len(X_test):,}")
print(f"Labels: {len(mlb.classes_)}")
display(genre_counts.head(20).to_frame("count"))

Rows used: 240
Train: 144 | Valid: 36 | Test: 60
Labels: 32


,count
genre_labels,
Romance,29762
Comedy,21282
Drama,18702
Fantasy,16125
Action,12734
BL,12667
School Life,12582
Yaoi,10131
Seinen,9235


## 4. Fine-tuning leger DistilBERT

On entraine une seule epoch avec de petits batches. Le but est de prouver le pipeline Transfer Learning, pas de chercher le meilleur score possible en local.

In [4]:
config = TransformerConfig(
    model_name="distilbert-base-uncased",
    max_length=128,
    learning_rate=2e-5,
    train_batch_size=8,
    eval_batch_size=16,
    epochs=1,
    threshold=0.5,
    output_dir=str(PROJECT_DIR / "models" / "transformer_novelforge"),
)

transformer = NovelForgeTransformer(
    num_labels=len(mlb.classes_),
    id2label=id2label,
    label2id=label2id,
    config=config,
)

start_time = time.perf_counter()
trainer = transformer.fine_tune(X_train, y_train, X_valid, y_valid)
training_seconds = time.perf_counter() - start_time
transformer.save(config.output_dir)
joblib.dump(list(mlb.classes_), PROJECT_DIR / "models" / "transformer_labels.joblib")

print(f"Transformer training time: {training_seconds:.1f} seconds")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,No log,0.604658,0.190476,0.047638


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Transformer training time: 28.1 seconds


## 5. Evaluation

On evalue le Transformer avec les memes metriques que la baseline : F1 micro, F1 macro, F1 weighted, Jaccard samples et Hamming loss.

In [5]:
metrics = transformer.evaluate(X_test, y_test, target_names=list(mlb.classes_), threshold=0.5)
summary = pd.Series(
    {
        "transformer_f1_micro": metrics["f1_micro"],
        "transformer_f1_macro": metrics["f1_macro"],
        "transformer_f1_weighted": metrics["f1_weighted"],
        "transformer_jaccard_samples": metrics["jaccard_samples"],
        "transformer_hamming_loss": metrics["hamming_loss"],
        "training_seconds": training_seconds,
    }
).to_frame("score")

display(summary.round(4))
print(metrics["classification_report_text"])

C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,score
transformer_f1_micro,0.1847
transformer_f1_macro,0.0449
transformer_f1_weighted,0.1189
transformer_jaccard_samples,0.0963
transformer_hamming_loss,0.1333
training_seconds,28.0858


               precision    recall  f1-score   support

       Action       0.00      0.00      0.00        12
    Adventure       0.00      0.00      0.00         5
           BL       0.32      0.67      0.43         9
       Comedy       1.00      0.07      0.12        15
       Demons       0.00      0.00      0.00         0
        Drama       0.25      0.46      0.32        13
      Fantasy       0.00      0.00      0.00        14
           GL       0.00      0.00      0.00         1
        Harem       0.00      0.00      0.00         3
   Historical       0.00      0.00      0.00         4
       Horror       0.05      1.00      0.10         3
       Isekai       0.00      0.00      0.00         1
        Josei       0.00      0.00      0.00         8
        Magic       0.00      0.00      0.00         1
 Martial Arts       0.00      0.00      0.00         2
        Mecha       0.00      0.00      0.00         0
      Mystery       0.00      0.00      0.00         6
Psycholog

## Conclusion Jalon 8

Le Transformer est l'approche la plus avancee du projet car il combine attention et Transfer Learning. Contrairement au TF-IDF, il exploite le contexte des tokens; contrairement au LSTM entraine depuis zero, il reutilise des representations linguistiques pre-entrainees. Sur CPU et petit echantillon, le score peut rester modeste, mais l'objectif du jalon est de demontrer un pipeline industriel realiste : chargement HuggingFace, fine-tuning multilabel, evaluation et sauvegarde du modele.

Pour une version finale performante, il faudrait augmenter le sous-echantillon, entrainer plus longtemps, calibrer les seuils par label, et comparer plusieurs modeles legers comme DistilBERT, MiniLM ou TinyBERT.